In [142]:
import pandas as pd

In [143]:
# Constants
FILE_IN = {"./Data/microbiology_cultures_cohort.csv": ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'ordering_mode', 'culture_description', 'was_positive', 'organism', 'antibiotic', 'susceptibility'], 
           "./Data/microbiology_culture_prior_infecting_organism.csv": ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'prior_organism', 'prior_infecting_organism_days_to_culutre'], 
           "./Data/microbiology_cultures_demographics.csv": ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'age', 'gender'],
           "./Data/microbiology_cultures_ward_info.csv": ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'hosp_ward_IP', 'hosp_ward_OP', 'hosp_ward_ER', 'hosp_ward_ICU']}

FILE_OUT = "./data_prepared.csv"
LEARNING_RATE = 0.1
ANTIBIOTICS = ["Ampicillin", "Cefazolin", "Ceftriaxone", "Ciprofloxacin", "Gentamicin", "Levofloxacin", "Meropenem", "Nitrofurantoin", "Vancomycin"]
SUSCEPTIBILITY = ['Susceptible', 'Resistant']
DROP_DUPLICATES = ['anon_id', 'order_proc_id_coded', 'organism', 'antibiotic']


# pandas settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [144]:
# The modifications are done to the main file first to avoid unnecessary memory usage.

# Read data
file = "./Data/microbiology_cultures_cohort.csv"
df = pd.read_csv(file)

# Original data
print("Original data")
print("Rows:", len(df.index))

# Drop NaN values
print("\nDrop NaN")
df_dropna = df[FILE_IN[file]].replace("Null", pd.NA).dropna(how="any")
print("Rows:", len(df_dropna.index))

# Filter unwanted susceptibility values
print("\nFilter susceptibility")
df_sus_rep = df_dropna[(df_dropna['susceptibility'].isin(SUSCEPTIBILITY))]
print("Rows:", len(df_sus_rep.index))

# Filter unwanted antibiotics
print("\nFilter antibiotics")
df_antibiotics = df_sus_rep[(df_sus_rep['antibiotic'].isin(ANTIBIOTICS))]
print("Rows:", len(df_antibiotics.index))



Original data
Rows: 2241050

Drop NaN
Rows: 1540541

Filter susceptibility
Rows: 1489839

Filter antibiotics
Rows: 708155


In [145]:
# Merging data files based on common rows. Also deletion of NaN values and 
unique_id = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc']
df = df_antibiotics

print("Merging prior infecting organism")
file = "./Data/microbiology_culture_prior_infecting_organism.csv"
df_in = pd.read_csv(file)
df_in = df_in[FILE_IN[file]].replace("Null", pd.NA).dropna(how="any")
df = pd.merge(df, df_in[FILE_IN[file]], on=unique_id)
print("rows:", len(df.index))

print()
print("Merging ward information")
file = "./Data/microbiology_cultures_ward_info.csv"
df_in = pd.read_csv(file)
df_in = df_in[FILE_IN[file]].replace("Null", pd.NA).dropna(how="any")
df = pd.merge(df, df_in[FILE_IN[file]], on=unique_id)
print("rows:", len(df.index))

print()
print("Merging demographics")
unique_id_2 = ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded']
file = "./Data/microbiology_cultures_demographics.csv"
df_in = pd.read_csv(file)
df_in = df_in[FILE_IN[file]].replace("Null", pd.NA).dropna(how="any")
df = pd.merge(df, df_in[FILE_IN[file]], on=unique_id_2)
print("rows:", len(df.index))

print(df)

Merging prior infecting organism
rows: 2213952

Merging ward information
rows: 2213952

Merging demographics


C:\Users\Emma\AppData\Local\Temp\ipykernel_3140\3931652944.py:24: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_in = pd.read_csv(file)


rows: 2213922
           anon_id  pat_enc_csn_id_coded  order_proc_id_coded    order_time_jittered_utc ordering_mode culture_description  was_positive               organism      antibiotic susceptibility  prior_organism  prior_infecting_organism_days_to_culutre  hosp_ward_IP  hosp_ward_OP  hosp_ward_ER  hosp_ward_ICU          age gender
0        JC2115956          131242371022            545978251  2018-01-13 12:35:00+00:00     Inpatient               URINE             1  KLEBSIELLA PNEUMONIAE       Meropenem    Susceptible  Staphylococcus                                        78             1             0             1              0  45-54 years      0
1        JC2115956          131242371022            545978251  2018-01-13 12:35:00+00:00     Inpatient               URINE             1  KLEBSIELLA PNEUMONIAE       Meropenem    Susceptible  Staphylococcus                                       153             1             0             1              0  45-54 years      0
2       

In [ ]:

# Drop duplicate rows
print("Drop duplicates")
df_drop_dupl = df.drop_duplicates(subset=DROP_DUPLICATES)
print("rows:", len(df_drop_dupl.index))

Drop duplicates
Size: 6115158
rows: 339731



In [ ]:
# Print info about the data
print("Time span:", df_drop_dupl['order_time_jittered_utc'].min(), "-", df_drop_dupl['order_time_jittered_utc'].max())
print("Columns:", df_drop_dupl.columns.values)

Time span: 2008-02-23 22:59:00+00:00 - 2024-01-30 21:49:00+00:00
Columns: ['anon_id' 'pat_enc_csn_id_coded' 'order_proc_id_coded'
 'order_time_jittered_utc' 'ordering_mode' 'culture_description'
 'was_positive' 'organism' 'antibiotic' 'susceptibility' 'prior_organism'
 'prior_infecting_organism_days_to_culutre' 'hosp_ward_IP' 'hosp_ward_OP'
 'hosp_ward_ER' 'hosp_ward_ICU' 'age' 'gender']


In [148]:
# Save prepared data into an excel
df_drop_dupl.to_csv(FILE_OUT, index=False)